In [ ]:
import os
import subprocess
import sys

import kagglehub
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM


print("=" * 60)
print("NEMOTRON-3 MOE ROUTER COLLAPSE ANALYSIS")
print("Analyzing expert routing weights for specialization vs collapse")
print("=" * 60)

try:
    # 1. Dependency Installation
    print("\n[1/5] Loading dependencies...")
    MANDATORY_PACKAGES = [
        "trl",
        "peft",
        "bitsandbytes",
        "accelerate",
        "nvidia-cutlass",
        "mamba_ssm",
        "causal_conv1d",
    ]
    for pkg in MANDATORY_PACKAGES:
        try:
            __import__(pkg.replace("-", "_"))
        except ImportError:
            print(f"  Installing {pkg}...")
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", pkg]
            )

    # 2. Blackwell Environment Setup (for mamba_ssm)
    print("\n[2/5] Setting up Blackwell environment...")
    UTILITY_PATH = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script"
    if os.path.exists(UTILITY_PATH):
        subprocess.run(f"tar -cf - -C {UTILITY_PATH} . | tar -xf - -C /tmp", shell=True, check=True)
        os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas-blackwell"
        sys.path.insert(0, "/tmp")

    # 3. Model Loading
    print("\n[3/5] Loading model weights...")
    model_id = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
    model_path = kagglehub.model_download(model_id)

    os.makedirs("/tmp/offload", exist_ok=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        offload_folder="/tmp/offload",
    )

    print("\n[4/5] Analyzing router weights across MoE layers...")
    results = []

    for i, layer in enumerate(model.model.layers):
        if hasattr(layer, "mlp") and hasattr(layer.mlp, "gate"):
            gate_weight = layer.mlp.gate.weight.detach().float().cpu().numpy()
            num_experts, hidden_size = gate_weight.shape

            expert_variances = np.var(gate_weight, axis=1)
            norms = np.linalg.norm(gate_weight, axis=1, keepdims=True)
            normalized_weights = gate_weight / (norms + 1e-8)
            cosine_sim_matrix = np.dot(normalized_weights, normalized_weights.T)
            np.fill_diagonal(cosine_sim_matrix, np.nan)

            avg_cosine_sim = np.nanmean(cosine_sim_matrix)
            max_cosine_sim = np.nanmax(cosine_sim_matrix)

            magnitudes = np.linalg.norm(gate_weight, axis=1)
            probs = magnitudes / np.sum(magnitudes)
            entropy = -np.sum(probs * np.log(probs + 1e-8))

            results.append(
                {
                    "layer": i,
                    "num_experts": int(num_experts),
                    "avg_cosine_sim": float(avg_cosine_sim),
                    "max_cosine_sim": float(max_cosine_sim),
                    "routing_entropy": float(entropy),
                    "min_variance": float(np.min(expert_variances)),
                    "max_variance": float(np.max(expert_variances)),
                }
            )
            print(f"Layer {i}: Avg Sim={avg_cosine_sim:.4f}, Entropy={entropy:.4f}")

    print("\n[5/5] Saving analysis results...")
    df = pd.DataFrame(results)
    df.to_csv("nemotron_router_analysis.csv", index=False)
    print(df.describe())

except Exception as e:
    import traceback

    print(f"\nERROR: {e}")
    traceback.print_exc()